In [7]:
import numpy as np
import time
import pandas as pd
from collections import OrderedDict
from spatialmath.base import rodrigues, tr2rpy

gripper = np.load("scalingforce/gripper_log.npy", allow_pickle=True).item()
robot   = np.load("scalingforce/robot_log.npy",   allow_pickle=True).item()
wrist   = np.load("scalingforce/wrist_log.npy",   allow_pickle=True).item()
wkspc   = np.load("scalingforce/wkspc_log.npy",   allow_pickle=True).item()

In [9]:
import numpy as np
from collections import OrderedDict

def downsample_robot_log(robot, target_hz=7, original_hz=47):
    """
    Downsample high-frequency robot log using a moving average filter.

    Args:
        robot (dict): Original high-frequency robot log, keyed by timestamp.
        target_hz (int): Desired downsampled frequency.
        original_hz (int): Original robot log frequency.

    Returns:
        OrderedDict: Downsampled robot log, same structure as original.
    """
    interval = 1 / target_hz  # e.g., 0.143s for 7Hz
    sorted_ts = sorted(robot.keys())
    ts_array = np.array(sorted_ts)

    # Generate downsampled target timestamps
    start_time = ts_array[0]
    end_time = ts_array[-1]
    target_times = []
    t = start_time
    while t <= end_time:
        target_times.append(t)
        t += interval

    robot_downsampled = OrderedDict()

    for t_target in target_times:
        # Use ±half interval for smoothing window
        window_start = t_target - interval / 2
        window_end = t_target + interval / 2
        window_mask = (ts_array >= window_start) & (ts_array < window_end)
        window_ts = ts_array[window_mask]

        if len(window_ts) == 0:
            continue  # skip if no data falls into the window

        # Get the entries in the window
        window_entries = [robot[t] for t in window_ts]

        # Moving average over the window
        q_avg = np.mean([e["actual_q"] for e in window_entries], axis=0)
        pose_avg = np.mean([e["actual_TCP_pose"] for e in window_entries], axis=0)
        speed_avg = np.mean([e["actual_TCP_speed"] for e in window_entries], axis=0)
        wrench_avg = np.mean([e["wrench"] for e in window_entries], axis=0)

        robot_downsampled[t_target] = {
            "actual_q": q_avg,
            "actual_TCP_pose": pose_avg,
            "actual_TCP_speed": speed_avg,
            "wrench": wrench_avg
        }

    # Recompute future TCP pose
    downsampled_ts = list(robot_downsampled.keys())
    for i in range(len(downsampled_ts) - 1):
        curr_ts = downsampled_ts[i]
        next_ts = downsampled_ts[i + 1]
        robot_downsampled[curr_ts]["future_TCP_pose"] = robot_downsampled[next_ts]["actual_TCP_pose"]
    # Last timestamp gets self as future
    last_ts = downsampled_ts[-1]
    robot_downsampled[last_ts]["future_TCP_pose"] = robot_downsampled[last_ts]["actual_TCP_pose"]

    return robot_downsampled

In [10]:
'''
robot log is a dict of <timestamp> keys and values of dicts consisting of the following 5 6D vectors:
    "actual_q": q,
    "actual_qd": qd,
    "actual_TCP_pose": pose,
    "actual_TCP_speed": speed,
    "wrench": ft_curr

i want to restructure it such that the dict values are just a flat list corresponding to 
'q0', 'q1', 'q2', 'q3', 'q4', 'q5', 'x', 'y', 'z', 'rx', 'ry', 'rz', 'dx', 'dy', 'dz', 'drx', 'dry', 'drz', vx, vy, vz, vrx, vry, vrz, fx, fy, fz, tx, ty, tz

here, we don't use actual_qd. q0-q5 corresponds to actual_q, 
x,y,z,rx,ry,rz corresponds to actual_TCP_pose, 
dx,dy,dz,drx,dry,drz corresponds to the delta between the current and the future TCP pose (the current shifted forward by 1)
vx,vy,vz,vrx,vry,vrz corresponds to actual_TCP_speed
fx,fy,fz,tx,ty,tz corresponds to wrench
'''
# Helper: convert 6D pose (xyz + axis-angle) to RPY
def axis_angle_to_rpy(pose):
    rot_matrix = axis_angle_to_rotation_mtrx(pose[3:])
    rpy = tr2rpy(rot_matrix, unit='rad')  # returns [roll, pitch, yaw]
    return np.concatenate([pose[:3], rpy])

# Helper: convert axis-angle to 3x3 rotation matrix
def axis_angle_to_rotation_mtrx(axis_angle):
    return rodrigues(axis_angle)  # spatialmath’s implementation

# Transform TCP speed from base frame to TCP frame
def transform_velocity_to_tcp_frame(tcp_pose, tcp_speed):
    R = axis_angle_to_rotation_mtrx(tcp_pose[3:])
    v_base = np.array(tcp_speed[:3])
    w_base = np.array(tcp_speed[3:])
    v_tcp = R.T @ v_base
    w_tcp = R.T @ w_base
    return np.concatenate((v_tcp, w_tcp))

# Your robot log (now called `robot`) is a dict of timestamp -> sensor dict
def flatten_robot_log(robot):
    timestamps = sorted(robot.keys())
    
    # Add future TCP pose
    for i in range(len(timestamps) - 1):
        curr_ts = timestamps[i]
        next_ts = timestamps[i + 1]
        robot[curr_ts]["future_TCP_pose"] = robot[next_ts]["actual_TCP_pose"]
    
    # Duplicate last pose
    last_ts = timestamps[-1]
    robot[last_ts]["future_TCP_pose"] = robot[last_ts]["actual_TCP_pose"]
    
    # Flatten everything
    robot_flat = OrderedDict()
    for ts in timestamps:
        entry = robot[ts]

        q = entry["actual_q"]  # [q0–q5]

        pose_rpy = axis_angle_to_rpy(entry["actual_TCP_pose"])  # [x, y, z, r, p, y]
        future_pose_rpy = axis_angle_to_rpy(entry["future_TCP_pose"])  # same
        delta_pose_rpy = future_pose_rpy - pose_rpy  # [dx, dy, dz, dr, dp, dy]

        tcp_speed_tcp_frame = transform_velocity_to_tcp_frame(
            entry["actual_TCP_pose"], entry["actual_TCP_speed"]
        )  # [vx, vy, vz, vrx, vry, vrz] in TCP frame

        wrench = entry["wrench"]  # [fx, fy, fz, tx, ty, tz]

        flat_vec = (
            list(q) +
            list(pose_rpy) +
            list(delta_pose_rpy) +
            list(tcp_speed_tcp_frame) +
            list(wrench)
        )

        robot_flat[ts] = flat_vec

    return robot_flat

# robot_flat_rpy = flatten_robot_log(robot)


In [11]:
def merge_logs_with_robot_flat(robot_flat, wrist_log, wkspc_log, gripper_log):
    from bisect import bisect_left

    def find_closest(ts_list, target):
        """Efficiently find the closest timestamp in a sorted list."""
        pos = bisect_left(ts_list, target)
        if pos == 0:
            return ts_list[0]
        if pos == len(ts_list):
            return ts_list[-1]
        before = ts_list[pos - 1]
        after = ts_list[pos]
        return before if abs(before - target) <= abs(after - target) else after

    # Pre-sort the timestamps from the slower logs
    wrist_ts = sorted(wrist_log.keys())
    wkspc_ts = sorted(wkspc_log.keys())
    gripper_ts = sorted(gripper_log.keys())

    merged = OrderedDict()

    for ts in robot_flat:
        closest_wrist_ts = find_closest(wrist_ts, ts)
        closest_wkspc_ts = find_closest(wkspc_ts, ts)
        closest_gripper_ts = find_closest(gripper_ts, ts)

        cf_l = gripper_log[closest_gripper_ts]['cf_l']
        cf_r = gripper_log[closest_gripper_ts]['cf_r']
        wrist_img = wrist_log[closest_wrist_ts]['rgb']
        wkspc_img = wkspc_log[closest_wkspc_ts]['rgb']

        # Add [cf_l, cf_r] and images at the end
        merged[ts] = {
            'data': robot_flat[ts] + [cf_l, cf_r],
            'wrist_img': wrist_img,
            'wkspc_img': wkspc_img
        }

    return merged


In [14]:
robot_ds = downsample_robot_log(robot, target_hz=15)
robot_flat = flatten_robot_log(robot_ds)
merged_log = merge_logs_with_robot_flat(robot_flat, wrist, wkspc, gripper)
len(merged_log)

45